# Test the best obtained WordDetector model

Inspect a trained `best_model.pth` checkpoint end-to-end, **fully offline**:

1. Export the checkpoint to the ONNX inference artifact + `config.json` (ADR 006).
2. Build a `WordDetectorModel` from the *local* files (no HF Hub needed).
3. Run detection on example images and visualise the predicted word boxes.
4. Cross-check the ONNX path against the raw torch checkpoint path.

Requires the `training-word-detector` extra. Run from the repo root with
`uv run jupyter lab`.

In [ ]:
import json
from pathlib import Path
from urllib.request import urlretrieve

import cv2
import matplotlib.pyplot as plt
import onnxruntime as ort

from xournalpp_htr.inference_models import WordDetectorModel
from xournalpp_htr.training.shared.bounding_box import BoundingBox
from xournalpp_htr.training.shared.postprocessing import draw_bboxes_on_image
from xournalpp_htr.training.word_detector.export import export
from xournalpp_htr.training.word_detector.infer import run_image_through_network
from xournalpp_htr.training.word_detector.utils import get_example_list

# Path to the trained checkpoint (gitignored; produced by train.py).
CHECKPOINT = Path("best_model.pth")
EXPORT_DIR = Path("exports")
assert CHECKPOINT.exists(), f"No checkpoint at {CHECKPOINT.resolve()}"

## 1. Export checkpoint -> ONNX + config.json

In [ ]:
paths = export(CHECKPOINT, EXPORT_DIR)
paths

## 2. Build a WordDetectorModel from the local export

`from_pretrained` would fetch from HF Hub; here we construct the same object
directly from the local ONNX + config so the notebook stays offline.

In [ ]:
with open(paths["config"]) as f:
    config = json.load(f)

model = WordDetectorModel(
    session=ort.InferenceSession(str(paths["onnx"])),
    config=config,
    revision="local",
)
model

## 3. Run detection on example images (PyTorch vs ONNX side-by-side)

In [ ]:
from xournalpp_htr.training.word_detector.network import WordDetectorNet

examples = get_example_list()  # list[str] of image URLs

for url in examples:
    local = Path(url.split("/")[-1])
    if not local.exists():
        urlretrieve(url, local)
    img_gray = cv2.imread(str(local), cv2.IMREAD_GRAYSCALE)

    # ONNX prediction (boxes already in original image coordinates)
    onnx_boxes = model.detect(img_gray)
    onnx_vis = draw_bboxes_on_image(img_gray, onnx_boxes, denormalise=False)

    # PyTorch prediction (boxes in network-input 448x448 space)
    torch_result = run_image_through_network(
        img_gray, model_path=CHECKPOINT, device="cpu"
    )
    torch_boxes = torch_result["aabbs"]

    # Rescale PyTorch boxes to original image coordinates
    orig_h, orig_w = img_gray.shape[:2]
    net_w, net_h = WordDetectorNet.input_size
    sx, sy = orig_w / net_w, orig_h / net_h
    torch_boxes_rescaled = [
        BoundingBox(b.x_min * sx, b.y_min * sy, b.x_max * sx, b.y_max * sy, b.label)
        for b in torch_boxes
    ]
    torch_vis = draw_bboxes_on_image(img_gray, torch_boxes_rescaled, denormalise=False)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 12))
    ax1.set_title(f"{local.name}: {len(torch_boxes)} words (PyTorch)")
    ax1.imshow(cv2.cvtColor(torch_vis, cv2.COLOR_BGR2RGB))
    ax1.axis("off")
    ax2.set_title(f"{local.name}: {len(onnx_boxes)} words (ONNX)")
    ax2.imshow(cv2.cvtColor(onnx_vis, cv2.COLOR_BGR2RGB))
    ax2.axis("off")
    plt.tight_layout()
    plt.show()